In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard imports
import numpy as np
import xarray as xr
import tqdm as tqdm

In [3]:
# Import OpenSense modules as submodules
import sys
import os

sys.path.append(os.path.abspath("./pycomlink/"))
sys.path.append(os.path.abspath("./poligrain/src/"))
sys.path.append(os.path.abspath("./mergeplg/src/"))

import pycomlink as pycml 
import poligrain as plg
import mergeplg 

In [4]:
os.makedirs('data/adjusted_fields', exist_ok=True)

In [5]:
# methods parameter : default version
nnear = 12 
variogram_parameters = {"sill": 0.8, "range": 30000, "nugget": 0.2}
diff_check_sel = 10
ratio_check_sel = (0.1,15)

# OpenMRG - Adjust rainfall fields

In [6]:
# OpenMRG
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")                    
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc")       
months = ['2015-06', '2015-07', '2015-08']                           


In [7]:
# additive IDW
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="additive",
        nnear=nnear,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_add_IDW_small": xr.concat(rainfall, dim="time") })
data['rainfall_add_IDW_small']=xr.where(
    np.isnan(data['rainfall_add_IDW_small']), 0, data['rainfall_add_IDW_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_add_IDW_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:13<00:00, 54.72it/s]


month: 2015-07


100%|██████████| 744/744 [00:12<00:00, 58.12it/s]


month: 2015-08


100%|██████████| 744/744 [00:12<00:00, 58.36it/s]


In [8]:
# multiplicative IDW
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="multiplicative",
        nnear=nnear,
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_mul_IDW_small": xr.concat(rainfall, dim="time") })
data['rainfall_mul_IDW_small']=xr.where(
    np.isnan(data['rainfall_mul_IDW_small']), 0, data['rainfall_mul_IDW_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_mul_IDW_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:12<00:00, 55.92it/s]


month: 2015-07


100%|██████████| 744/744 [00:13<00:00, 56.46it/s]


month: 2015-08


100%|██████████| 744/744 [00:13<00:00, 56.87it/s]


In [9]:
# additive POINT BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_add_pBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_add_pBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_add_pBK_stnd_small']), 0, data['rainfall_add_pBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_add_pBK_stnd_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:22<00:00, 31.85it/s]


month: 2015-07


100%|██████████| 744/744 [00:29<00:00, 25.59it/s]


month: 2015-08


100%|██████████| 744/744 [00:23<00:00, 31.25it/s]


In [10]:
# multiplicative POINT BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_mul_pBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_mul_pBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_mul_pBK_stnd_small']), 0, data['rainfall_mul_pBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_mul_pBK_stnd_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:19<00:00, 36.99it/s] 


month: 2015-07


100%|██████████| 744/744 [00:22<00:00, 32.64it/s] 


month: 2015-08


100%|██████████| 744/744 [00:18<00:00, 39.30it/s] 


In [11]:
# additive LINE BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_add_lBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_add_lBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_add_lBK_stnd_small']), 0, data['rainfall_add_lBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_add_lBK_stnd_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:22<00:00, 31.61it/s] 


month: 2015-07


100%|██████████| 744/744 [00:26<00:00, 27.86it/s] 


month: 2015-08


100%|██████████| 744/744 [00:24<00:00, 30.58it/s] 


In [12]:
# multiplicative LINE BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_mul_lBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_mul_lBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_mul_lBK_stnd_small']), 0, data['rainfall_mul_lBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_mul_lBK_stnd_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:19<00:00, 36.41it/s]


month: 2015-07


100%|██████████| 744/744 [00:23<00:00, 31.50it/s] 


month: 2015-08


100%|██████████| 744/744 [00:18<00:00, 39.65it/s] 


In [13]:
# KED point
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_ked_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_ked_stnd_small']=xr.where(
    np.isnan(data['rainfall_ked_stnd_small']), 0, data['rainfall_ked_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_pked_stnd_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:24<00:00, 29.78it/s] 


month: 2015-07


100%|██████████| 744/744 [00:31<00:00, 23.66it/s]


month: 2015-08


100%|██████████| 744/744 [00:24<00:00, 30.59it/s]


In [14]:
# KED line
variogram_parameters = {"sill": 1.0, "range": 30000, "nugget": 0.1}
nnear = 12
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_ked_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_ked_stnd_small']=xr.where(
    np.isnan(data['rainfall_ked_stnd_small']), 0, data['rainfall_ked_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_lked_stnd_small.nc')    
del data, merger

month: 2015-06


100%|██████████| 719/719 [00:24<00:00, 29.70it/s] 


month: 2015-07


100%|██████████| 744/744 [00:29<00:00, 25.08it/s] 


month: 2015-08


100%|██████████| 744/744 [00:26<00:00, 28.09it/s]


# OpenRainER

In [15]:
# OpenRainER
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")         
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc")   
ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        


# Adjust rainfall fields

In [16]:
os.makedirs('data/adjusted_fields', exist_ok=True)
months = ['2022-06', '2022-07', '2022-08']

In [ ]:
# additive IDW
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="additive",
        nnear=nnear,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_add_IDW_small": xr.concat(rainfall, dim="time") })
data['rainfall_add_IDW_small']=xr.where(
    np.isnan(data['rainfall_add_IDW_small']), 0, data['rainfall_add_IDW_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_add_IDW_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [00:48<00:00, 14.73it/s]


month: 2022-07


100%|██████████| 744/744 [00:50<00:00, 14.87it/s]


month: 2022-08


100%|██████████| 744/744 [00:51<00:00, 14.43it/s]


In [18]:
# multiplicative IDW
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="multiplicative",
        nnear=nnear,
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_mul_IDW_small": xr.concat(rainfall, dim="time") })
data['rainfall_mul_IDW_small']=xr.where(
    np.isnan(data['rainfall_mul_IDW_small']), 0, data['rainfall_mul_IDW_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_mul_IDW_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [00:49<00:00, 14.54it/s]


month: 2022-07


100%|██████████| 744/744 [00:49<00:00, 15.09it/s]


month: 2022-08


100%|██████████| 744/744 [00:51<00:00, 14.55it/s]


In [19]:
# additive POINT BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_add_pBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_add_pBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_add_pBK_stnd_small']), 0, data['rainfall_add_pBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_add_pBK_stnd_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [11:05<00:00,  1.08it/s]


month: 2022-07


100%|██████████| 744/744 [02:42<00:00,  4.59it/s]


month: 2022-08


100%|██████████| 744/744 [08:16<00:00,  1.50it/s]


In [20]:
# multiplicative POINT BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_mul_pBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_mul_pBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_mul_pBK_stnd_small']), 0, data['rainfall_mul_pBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_mul_pBK_stnd_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [06:00<00:00,  1.99it/s]


month: 2022-07


100%|██████████| 744/744 [01:49<00:00,  6.83it/s] 


month: 2022-08


100%|██████████| 744/744 [05:27<00:00,  2.27it/s]


In [21]:
# additive LINE BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_add_lBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_add_lBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_add_lBK_stnd_small']), 0, data['rainfall_add_lBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_add_lBK_stnd_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [10:19<00:00,  1.16it/s]


month: 2022-07


100%|██████████| 744/744 [02:39<00:00,  4.66it/s]


month: 2022-08


100%|██████████| 744/744 [07:59<00:00,  1.55it/s]


In [22]:
# multiplicative LINE BLOCK KRIGING
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_mul_lBK_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_mul_lBK_stnd_small']=xr.where(
    np.isnan(data['rainfall_mul_lBK_stnd_small']), 0, data['rainfall_mul_lBK_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_mul_lBK_stnd_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [05:43<00:00,  2.09it/s]


month: 2022-07


100%|██████████| 744/744 [01:44<00:00,  7.10it/s]


month: 2022-08


100%|██████████| 744/744 [05:27<00:00,  2.27it/s]


In [23]:
# KED point
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_ked_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_ked_stnd_small']=xr.where(
    np.isnan(data['rainfall_ked_stnd_small']), 0, data['rainfall_ked_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_pked_stnd_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [10:47<00:00,  1.11it/s]


month: 2022-07


100%|██████████| 744/744 [02:41<00:00,  4.62it/s]


month: 2022-08


100%|██████████| 744/744 [08:00<00:00,  1.55it/s]


In [24]:
# KED line
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"rainfall_ked_stnd_small": xr.concat(rainfall, dim="time") })
data['rainfall_ked_stnd_small']=xr.where(
    np.isnan(data['rainfall_ked_stnd_small']), 0, data['rainfall_ked_stnd_small']
    )
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_lked_stnd_small.nc')    
del data, merger

month: 2022-06


100%|██████████| 719/719 [11:02<00:00,  1.08it/s]


month: 2022-07


100%|██████████| 744/744 [02:38<00:00,  4.69it/s]


month: 2022-08


100%|██████████| 744/744 [08:21<00:00,  1.48it/s]
